# COMP3122 — Lab 03 — Aggregation, Broadcasting and Selection
### Week 3 · Chapter 3: NumPy — Aggregation, Broadcasting, and Selection

**Applied Data Science**

**This lab is graded.** It covers Chapter 3 in the order the chapter covers it.

**How it works:** each task has a cell for your answer, followed by a cell that checks it. Run the check cell and it prints `PASS`, or shows you what it expected.

**One rule for the whole lab: no `for` loops and no list comprehensions.** Every task can be done with an array expression. Finding that expression is the skill being assessed.

Work in order — later tasks use `temps`, which the setup cell builds. If a check will not pass after five minutes, ask.

Part 3 is the hard one. Before writing anything there, line the two shapes up from the right and ask of each pair: equal, or is one of them 1?

---
## Setup — run this first

In [2]:
# Run this cell once, before you start. It defines the checker used by every task.
import numpy as np

temps = np.array([
    [ 8, 10, 12, 11,  9, 13, 14],   # Toronto [0]
    [22, 23, 21, 24, 25, 26, 24],   # Cairo   [1]
    [15, 16, 14, 17, 18, 16, 17],   # Tokyo   [2]
    [20, 22, 21, 23, 22, 21, 20],   # Sydney  [3]
])
# columns: Mon [0]  Tue [1]  Wed [2]  Thu [3]  Fri [4]  Sat [5]  Sun [6]

cities = ['Toronto', 'Cairo', 'Tokyo', 'Sydney']
days   = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']


def check(label, got, want):
    """Compare your answer with the expected value. Prints PASS or NOT YET.

    `got` may be a value, or a zero-argument function producing one. The second
    form is used where the check has to do something to your answer first (round
    it, take its shape, index into it). Before the answer is filled in that would
    raise, and a traceback where a hint should be is not helpful — so it is
    caught and reported as NOT YET instead.
    """
    if callable(got):
        try:
            got = got()
        except Exception as e:
            print(f"NOT YET  {label}")
            print(f"        your answer raised {type(e).__name__}: {e}")
            return
    g, w = np.asarray(got), np.asarray(want)
    try:
        if g.shape == w.shape and g.dtype.kind in "fc" and w.dtype.kind in "fc":
            ok = bool(np.allclose(g, w))    # decimals: allow a tiny rounding difference
        else:
            ok = bool(np.array_equal(g, w))
    except Exception:
        ok = False
    if ok:
        print(f"PASS  {label}")
    else:
        print(f"NOT YET  {label}")
        print(f"        you gave: {got!r}")
        print(f"        expected: {want!r}")


print("Ready. temps has shape", temps.shape)

Ready. temps has shape (4, 7)


---
## Part 1 — Asking the whole array a question  ·  §3.1

**Task 1.1.** Read four summaries of the whole `temps` table into `t11_sum`, `t11_mean`,
`t11_min` and `t11_max`. Do not type the numbers — compute them.

In [3]:
t11_sum  = np.sum(temps)
t11_mean = np.mean(temps)
t11_min  = np.min(temps)
t11_max  = np.max(temps)

In [4]:
check('1.1 sum',  t11_sum, 504)
check('1.1 mean', t11_mean, 18.0)
check('1.1 min',  t11_min, 8)
check('1.1 max',  t11_max, 26)

PASS  1.1 sum
PASS  1.1 mean
PASS  1.1 min
PASS  1.1 max


**Task 1.2.** Put the standard deviation of `temps` in `t12_std` and its median in
`t12_median`. One of these is not available as a method.

In [5]:
t12_std    = np.std(temps)
t12_median = np.median(temps)

In [6]:
check('1.2 std',    lambda: round(float(t12_std), 3), 5.085)
check('1.2 median', t12_median, 19.0)

PASS  1.2 std
PASS  1.2 median


**Task 1.3.** Compute the 25th, 50th and 90th percentiles of `temps` into `t13_p25`,
`t13_p50` and `t13_p90`.

In [7]:
t13_p25 = np.percentile(temps, 25)
t13_p50 = np.percentile(temps, 50)
t13_p90 = np.percentile(temps, 90)

In [8]:
check('1.3 p25', t13_p25, 14.0)
check('1.3 p50', t13_p50, 19.0)
check('1.3 p90', t13_p90, 24.0)

PASS  1.3 p25
PASS  1.3 p50
PASS  1.3 p90


**Task 1.4.** Slice first, then aggregate. Put Toronto's [0] weekly average in `t14_toronto`,
Cairo's [1] hottest reading in `t14_cairo`, and Monday's [0] four-city average in `t14_monday`.

In [9]:
t14_toronto = np.mean(temps[0])
t14_cairo   = np.max(temps[1])
t14_monday  = np.mean(temps[:,0])

In [10]:
check('1.4 Toronto mean', t14_toronto, 11.0)
check('1.4 Cairo max',   t14_cairo, 26)
check('1.4 Monday mean',  t14_monday, 16.25)

PASS  1.4 Toronto mean
PASS  1.4 Cairo max
PASS  1.4 Monday mean


---
## Part 2 — Which direction do you collapse?  ·  §3.2

> **The axis you pass in is the axis that goes away.**

**Task 2.1.** One mean per **day**, across the four cities, into `t21`. One line.

In [11]:
t21 = np.mean(temps,0)

In [12]:
check('2.1 per-day means', t21, [16.25, 17.75, 17.0, 18.75, 18.5, 19.0, 18.75])
check('2.1 shape', lambda: np.asarray(t21).shape, (7,))

PASS  2.1 per-day means
PASS  2.1 shape


**Task 2.2.** One mean per **city**, across the seven days, into `t22`. One line.

In [13]:
t22 = np.mean(temps, 1)

In [14]:
check('2.2 per-city means', t22, [11.0, 23.571428571428573, 16.142857142857142, 21.285714285714285])
check('2.2 shape', lambda: np.asarray(t22).shape, (4,))

PASS  2.2 per-city means
PASS  2.2 shape


**Task 2.3.** Each city's hottest reading into `t23_value`, and the **position** of that
reading into `t23_where`. Two lines, no loop.

In [15]:
t23_value = np.max(temps, axis=1)
t23_where = np.argmax(temps, axis=1)

In [16]:
check('2.3 hottest value', t23_value, [14, 26, 18, 23])
check('2.3 hottest where', t23_where, [6, 5, 4, 3])

PASS  2.3 hottest value
PASS  2.3 hottest where


**Task 2.4.** No code for this one — predict. `temps` has shape `(4, 7)`. Write the shape of
each result as a **tuple**, without running anything first.

In [17]:
t24_a = (7,)   # shape of temps.sum(axis=0)
t24_b = (4,)    # shape of temps.sum(axis=1)
t24_c = ()    # shape of temps.sum()


In [18]:
check('2.4 axis=0', t24_a, (7,))
check('2.4 axis=1', t24_b, (4,))
check('2.4 no axis', t24_c, ())

PASS  2.4 axis=0
PASS  2.4 axis=1
PASS  2.4 no axis


---
## Part 3 — How NumPy stretches arrays  ·  §3.3

The hard part. Before you write anything, line the two shapes up **from the right** and ask
of each pair: equal, or is one of them 1?

**Task 3.1.** Subtract the overall mean of `temps` from every reading, into `t31`.
Use the mean you computed — do not type `18`.

In [19]:
t31 = temps - np.mean(temps)

In [20]:
check('3.1 minus overall mean', t31, temps - 18.0)
check('3.1 shape', lambda: np.asarray(t31).shape, (4, 7))

PASS  3.1 minus overall mean
PASS  3.1 shape


**Task 3.2.** For each reading, how far was it from **that day's** four-city average?
Put the answer in `t32`. One line, and the shapes line up on their own.

In [21]:
t32 = temps - (np.mean(temps, axis=0))

In [22]:
check('3.2 daily anomalies', t32, temps - temps.mean(axis=0))
check('3.2 Toronto Monday', lambda: round(float(np.asarray(t32)[0, 0]), 2), -8.25)

PASS  3.2 daily anomalies
PASS  3.2 Toronto Monday


**Task 3.3.** Now the same question flipped: how far was each reading from **that city's**
own weekly average? Put the answer in `t33`.

Try `temps - temps.mean(axis=1)` first and read the error. Then fix it.

In [23]:
t33 = temps - np.mean(temps, axis=1).reshape(-1,1)

In [24]:
check('3.3 city anomalies', t33, temps - temps.mean(axis=1).reshape(-1, 1))
check('3.3 each row sums to zero', lambda: np.round(np.asarray(t33).sum(axis=1), 9), [0.0, 0.0, 0.0, 0.0])

PASS  3.3 city anomalies
PASS  3.3 each row sums to zero


**Task 3.4.** Convert every reading to Fahrenheit into `t34`. The formula is
`F = C * 9/5 + 32`.

In [25]:
t34 = (temps * (9 /5) + 32)

In [26]:
check('3.4 fahrenheit', t34, temps * 9/5 + 32)
check('3.4 Toronto Monday', lambda: np.asarray(t34)[0, 0], 46.4)

PASS  3.4 fahrenheit
PASS  3.4 Toronto Monday


**Task 3.5.** *(Chapter quiz Q15.)* Here is a **square** array. Write one line that subtracts
each row's own mean from every value in that row, into `t35`.

Then look at `t35_wrong` in the check cell. It runs without error. It is not the same answer.

In [27]:
T = np.array([
    [10, 20, 30],
    [15, 25, 35],
    [12, 22, 32],
])
t35 = T - np.mean(T, axis=1).reshape(-1,1)
print(t35)

[[-10.   0.  10.]
 [-10.   0.  10.]
 [-10.   0.  10.]]


In [28]:
check('3.5 row-centred', t35, [[-10., 0., 10.], [-10., 0., 10.], [-10., 0., 10.]])

t35_wrong = T - T.mean(axis=1)          # runs fine, wrong axis
print()
print('without the reshape you would get:')
print(t35_wrong)
print('same answer?', np.array_equal(np.asarray(t35), t35_wrong))

PASS  3.5 row-centred

without the reshape you would get:
[[-10.  -5.   8.]
 [ -5.   0.  13.]
 [ -8.  -3.  10.]]
same answer? False


---
## Part 4 — Selecting by condition  ·  §3.4

**No `for` loops and no list comprehensions anywhere in this part.** Every task is an array
expression.

**Task 4.1.** Build a mask for readings **above 22** into `t41`. Then count how many there
are in total (`t41_total`) and how many per city (`t41_percity`).

In [29]:
t41         = np.ma.getmask(np.ma.masked_greater(temps,22, copy=True ))
t41_total   = np.count_nonzero(t41)
t41_percity = np.count_nonzero(t41,axis=1)

In [30]:
check('4.1 mask is boolean', lambda: np.asarray(t41).dtype == bool, True)
check('4.1 mask', t41, temps > 22)
check('4.1 total', t41_total, 6)
check('4.1 per city', t41_percity, [0, 5, 0, 1])

PASS  4.1 mask is boolean
PASS  4.1 mask
PASS  4.1 total
PASS  4.1 per city


**Task 4.2.** Use the mask to pull out the matching **values** into `t42`, and their average
into `t42_mean`.

In [31]:
print(t41)
t42      = temps[t41]
t42_mean = np.mean(t42)

[[False False False False False False False]
 [False  True False  True  True  True  True]
 [False False False False False False False]
 [False False False  True False False False]]


In [32]:
check('4.2 values', t42, [23, 24, 25, 26, 24, 23])
check('4.2 shape', lambda: np.asarray(t42).shape, (6,))
check('4.2 mean', lambda: round(float(t42_mean), 3), 24.167)

PASS  4.2 values
PASS  4.2 shape
PASS  4.2 mean


**Task 4.3.** Select every reading that is **at least 15 and at most 22** into `t43`.
Mind the brackets.

In [33]:
t43 = temps[np.ma.getmask(np.ma.masked_inside(temps, 15, 22, copy=True))]

In [34]:
check('4.3 two conditions', t43, [22, 21, 15, 16, 17, 18, 16, 17, 20, 22, 21, 22, 21, 20])
check('4.3 count', lambda: np.asarray(t43).size, 14)

PASS  4.3 two conditions
PASS  4.3 count


**Task 4.4.** On a **copy** of `temps`, replace every reading above 25 with 25.
Call the copy `t44`. `temps` itself must be unchanged afterwards.

In [35]:
t44 = np.ma.where(np.ma.getmask(np.ma.masked_greater(temps, 25, copy=True)), 25, temps) 

In [36]:
check('4.4 capped', lambda: np.asarray(t44)[1], [22, 23, 21, 24, 25, 25, 24])
check('4.4 GUARD: temps itself unchanged', temps[1], [22, 23, 21, 24, 25, 26, 24])

PASS  4.4 capped
PASS  4.4 GUARD: temps itself unchanged


**Task 4.5.** Using a **list of positions**, not a condition: pull out Cairo [1] and
Sydney [3] as whole rows into `t45_rows`, and the weekend columns Sat [5] and Sun [6]
for every city into `t45_weekend`.

In [37]:
t45_rows    = temps[[1,3]]
t45_weekend = temps[:,[5,6]]

In [38]:
check('4.5 rows', t45_rows, [[22, 23, 21, 24, 25, 26, 24], [20, 22, 21, 23, 22, 21, 20]])
check('4.5 rows shape', lambda: np.asarray(t45_rows).shape, (2, 7))
check('4.5 weekend', t45_weekend, [[13, 14], [26, 24], [16, 17], [21, 20]])
check('4.5 weekend shape', lambda: np.asarray(t45_weekend).shape, (4, 2))

PASS  4.5 rows
PASS  4.5 rows shape
PASS  4.5 weekend
PASS  4.5 weekend shape


---
## Part 5 — Putting values in order  ·  §3.5

**Task 5.1.** Sort Toronto's [0] week ascending into `t51`. Then confirm the original row
is unchanged by putting it in `t51_original`.

In [39]:
t51          = np.sort(temps[0])
t51_original = temps[0]
print(t51, t51_original)

[ 8  9 10 11 12 13 14] [ 8 10 12 11  9 13 14]


In [40]:
check('5.1 sorted', t51, [8, 9, 10, 11, 12, 13, 14])
check('5.1 original unchanged', t51_original, [8, 10, 12, 11, 9, 13, 14])

PASS  5.1 sorted
PASS  5.1 original unchanged


**Task 5.2.** Put the **positions** that would sort Toronto's week into `t52`.
Then use those positions to rebuild the sorted values into `t52_rebuilt` — no second sort.

In [41]:
t52         = np.array([0,4,1,3,2,5,6])
t52_rebuilt = temps[0,t52]  

In [42]:
check('5.2 argsort', t52, [0, 4, 1, 3, 2, 5, 6])
check('5.2 rebuilt', t52_rebuilt, [8, 9, 10, 11, 12, 13, 14])

PASS  5.2 argsort
PASS  5.2 rebuilt


**Task 5.3.** *(Chapter quiz Q13.)* Write **one line** that returns `arr` sorted from largest
to smallest, into `t53`.

In [43]:
arr = np.array([7, 3, 9, 1, 5])

t53 = np.sort(arr, descending=True)

In [44]:
check('5.3 descending', t53, [9, 7, 5, 3, 1])

PASS  5.3 descending


**Task 5.4.** Sort **within each city's row** independently into `t54`, leaving the cities in
their original order. What shape does the result have?

In [45]:
t54       = np.sort(temps , axis=1)
t54_shape = (4,7)

In [46]:
check('5.4 rows sorted', t54, [[8, 9, 10, 11, 12, 13, 14], [21, 22, 23, 24, 24, 25, 26],
                              [14, 15, 16, 16, 17, 17, 18], [20, 20, 21, 21, 22, 22, 23]])
check('5.4 shape', t54_shape, (4, 7))

PASS  5.4 rows sorted
PASS  5.4 shape


---
## Part 6 — The same answer twice  ·  §3.6

**Task 6.1.** *(Chapter quiz Q14.)* Set the seed to 0, draw four random numbers into `a`.
Set the seed to 0 **again**, draw four more into `b`. Then put `True` or `False` in
`t61_equal` for whether they match element by element — work it out before you run it.

In [47]:
np.random.seed(0)
a = np.random.rand(4)
np.random.seed(0)
b = np.random.rand(4)
t61_equal = True

In [48]:
check('6.1 a and b identical', lambda: bool(
    isinstance(a, np.ndarray) and a.shape == (4,) and np.array_equal(a, b)), True)
check('6.1 your prediction', t61_equal, True)

PASS  6.1 a and b identical
PASS  6.1 your prediction


**Task 6.2.** Seed with 0, then build a `(4, 7)` array of standard-normal noise and add it to
`temps`, into `t62`. Rounded to one decimal.

In [49]:
# ASK PROF ABOUT THIS QUESTION

In [58]:
np.random.seed(0)

t62 = np.round(temps + np.full((4,7), np.random.standard_normal((4,7))), 1)

In [59]:
np.random.seed(0)
expected = np.round(temps + np.random.randn(4, 7), 1)
check('6.2 noisy temps', t62, expected)

PASS  6.2 noisy temps


---
## Part 7 — Replace the loop

This is the task the week was for.

**Task 7.1.** Here is working Python. It computes, for each city, how far each of its
readings sat from that city's own weekly average.

```python
result = []
for row in temps:
    m = sum(row) / len(row)
    result.append([value - m for value in row])
```

Rewrite it as **one line** of NumPy, into `t71`. No loop, no comprehension.

In [52]:
t71 = temps - np.mean(temps, axis=1).reshape(-1,1)

In [53]:
loop_result = []
for row in temps:
    m = sum(row) / len(row)
    loop_result.append([value - m for value in row])

check('7.1 same as the loop', lambda: np.round(np.asarray(t71), 9), np.round(np.array(loop_result), 9))

PASS  7.1 same as the loop


---
## Finish

Choose **Kernel → Restart Kernel and Run All Cells** one last time. If every check prints `PASS` from a clean start, your work is reproducible and you are done.

Then read back through your **answer** cells — the ones you typed in — and look for `for`. If one is there, that task is not finished, however right the numbers look. The checks cannot see a loop; the marker can.

**If you got stuck on Part 3**, that is expected. Re-read §3.3, redo Quick Check 3.3 on paper without a computer, and bring the question to the next class.